
# 06 · FC-STEM (cepstral) — 약한 신호·혼합 비정질상 매핑

`RDF(structure factor) 방법`이 신호가 약해 잘 안 될 때의 대안. **EWPC(Exit-Wave Power Cepstrum)**:
$$\mathrm{EWPC}(\mathbf r)=\big|\,\mathcal F^{-1}\{\log I(\mathbf k)\}\,\big|$$
- **log** 이 다이내믹 레인지를 압축 → 약한 산란도 살아남음
- **배경 제거·structure factor 불필요** (실패하던 그 단계를 건너뜀)
- quefrency 축 = 실공간 거리(Å), 피크 = **원자간 거리** (RDF 유사)
- **Fluctuation(정규분산)** $F(R_p)=\langle C_p^2\rangle/\langle C_p\rangle^2-1$ 을 거리 밴드별로 → **혼합상 매핑**

> 참고: Pidaparthy, Ni, Hou, Abraham, Zuo, *Ultramicroscopy* **248** (2023) 113718.
> 켑스트럼은 모듈러스라 **비음수** → NMF도 유효(RDF의 부호 문제 없음).
> 모든 그림 PNG + 그래프 CSV는 각 셀에서 `SAVE_DIR`에 저장됩니다.


## 1) dm4 불러오기 & 설정

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)

DET_BIN    = 2            # 검출기 비닝(메모리/속도). 합성이면 1
Q_UNIT_HINT= "1/nm"     # ★ dm4 단위가 1/nm (0.043888 1/nm/px). 1/A로 두면 거리 10배 틀림
Q_PER_PX   = 0.02         # ★합성데이터 전용 fallback(1/A). 실데이터(dm4)는 이 값 무시하고
                          #   메타데이터(0.043888 1/nm →/10×binning= 0.008778 1/A)를 자동 사용.
                          #   여기에 0.043888을 넣지 마세요(1/nm≠1/A, 10배 틀림).
N_JOBS     = 4            # 병렬 코어수. ★메모리 부족(커널 꺼짐)이면 2로(-1/-2=전코어=메모리多)
EWPC_OFFSET= 1.0          # log(I+offset) — log(0) 방지
WINDOW     = True         # 켑스트럼 전 Hann 창(가장자리 십자 아티팩트 억제)
K          = 3            # 켑스트럼 프로파일 NMF/PCA 성분 수 (elbow 스캔으로 확인 후 조정)
KMAX_SCAN  = 8            # 성분 수 스캔 범위(1..KMAX): elbow/누적분산으로 '진짜 몇 개'인지
CENTER     = None         # (cx,cy) 수동. None이면 무게중심
CENTER_ZOOM= 25           # 중심 확대 그림 반경(px). 중심 맞는지 확인용(작을수록 확대)
STRONG_FRAC= 0.20         # §2d: 산란 강한 상위 비율만 골라 정렬평균(약한 링 추출 시도)
GAMMA      = 0.35         # §2d 2D 표시 감마(<1이면 약한 링 강조; 사용자가 감마로 링 본 것 재현)
RING_Q_MANUAL = None      # 감마 이미지에서 링이 보이면 그 q(1/A)를 직접 입력 → calibration/분석에 사용
INCLUDE_SUBSTRATE = False # 기판에 Cu가 있으면 True(Cu/Cu2O/CuO 후보 포함). 이 시료는 Cu 없음 → False
SIG_Q      = 0.045        # 링 패턴 언믹싱용 q-공간 브로드닝(1/A)
# 알려진 Li 화합물 원자간 거리(Å, 결정 근사) — 켑스트럼 프로파일에 참고선(비정질은 다소 이동)
CEPSTRAL_REF = {"Li-F 2.01": 2.01, "Li-O 2.00": 2.00, "Li-N 1.94": 1.94,
                "Li-S 2.47": 2.47, "2nd~2.9": 2.9}   # 후보 5개의 최근접이웃(pair)만
# 영역별 RDF용 설정 (조성은 peak 위치엔 영향 적음 — 아는 원소로 대략)
CFG_RDF = fds.RDFConfig(composition={"Li":1,"O":1}, q_int_min=0.15, q_int_max=1.0,
                        r_min=1.0, r_max=8.0, dr=0.02, damping="lorch")
# ★ 캘리브레이션 고정(known-standard): 비정질 첫 링(FSDP)의 최근접이웃 거리를 아는 값으로 맞춤.
#   None이면 메타데이터 q_per_px 그대로 사용(가정 없음).
#   Li-음이온이 주성분이면 CALIB_R_TARGET=2.0 (Å)로 두면 RDF 첫 peak이 그 거리로 옴.
#   → 메타데이터 대비 큰 보정이 필요하면 카메라 길이/단위를 다시 확인하세요.
CALIB_R_TARGET = None     # None=메타데이터(dm 0.043888 1/nm) 그대로 신뢰(권장). 아는 거리로 강제할 때만 값(예 2.0)
EHRENFEST      = 1.23     # cepstral 간격→pair 근사(단원자/금속글래스 기준; 이온성 Li는 1.1~1.4로 변함). 비율엔 무관, 절대거리는 RDF가 정확
# --- 빈(진공) 위치 제외 ---
# ★ nb5와 마스크를 '똑같이' 하려면 EMPTY_ROWS/EMPTY_ROI를 nb5와 같은 값으로 두세요.
#   (지정하면 그 빈 영역의 산란 평균+3σ로 임계 = nb5와 동일 방법. None이면 자동 Otsu라 조금 다름)
MATERIAL_MASK = True      # 물질 없는(진공) 위치를 분석에서 제외
ERODE_EDGE    = 2         # 물질 마스크를 이만큼(px) 침식 → 얇은 '가장자리 상'(두께 효과) 제외, 벌크만 분석
HOT_THRESHOLD = 8.0       # 고정 hot/dead 픽셀 검출 민감도(작을수록 민감)
DENOISE_CUBE  = False     # True면 전체 큐브의 고정 bad 픽셀 수리(메모리 큼). False면 평균 패턴만 정리
EMPTY_ROWS    = 10        # 아래 N행이 빈 영역(임계 기준). nb5의 EMPTY_ROWS와 같게 맞추세요
EMPTY_ROI     = None      # (y0,y1,x0,x1)로 빈 영역 직접 지정(있으면 EMPTY_ROWS 무시)
# 거리 밴드(Å) — 각각 하나의 FC-STEM 이미지. 데이터에 맞게 조절(먼저 §3 프로파일 보고).
BANDS      = [(1.0,1.5),(1.5,2.0),(2.0,2.5),(2.5,3.0),(3.0,3.5),(3.5,4.0),(4.0,4.5),(4.5,5.0),(5.0,5.5),(5.5,6.0)]

def make_fcstem_cube(scan=(36,48), dp=(96,96), empty_rows=8, seed=0):
    '''합성: 좌=결정(스팟), 중=비정질(halo), 우=다른 비정질(halo2), 아래=빈영역.
    FC-STEM이 결정/비정질/빈영역을 구분하는지 확인용.'''
    rng=np.random.default_rng(seed); Sy,Sx=scan; H,W=dp
    yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/(2*2.0**2))
    def halo(r0,s=4.0): return np.exp(-(rr-r0)**2/(2*s**2))
    def spots(r0,n=6,s=1.6,amp=4.0):
        img=np.zeros((H,W))
        for k in range(n):
            a=2*np.pi*k/n; sx,sy=cx+r0*np.cos(a),cy+r0*np.sin(a)
            img+=amp*np.exp(-((xx-sx)**2+(yy-sy)**2)/(2*s**2))
        return img
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            if iy>=Sy-empty_rows: base=0.9*beam                      # 빈 영역(진공)
            elif ix<Sx//3:        base=beam+spots(22)+0.5*halo(22)   # 결정
            elif ix<2*Sx//3:      base=beam+1.2*halo(20)             # 비정질 A
            else:                 base=beam+1.2*halo(28)             # 비정질 B
            cube[iy,ix]=base+0.15*rng.standard_normal((H,W))
    return np.clip(cube,0,None)

if USE_SYNTHETIC:
    cube=fds.from_array(make_fcstem_cube(empty_rows=(EMPTY_ROWS or 8)), q_per_px=Q_PER_PX, name="synthetic-FCSTEM")
else:
    cube=fds.load(DM4_PATH, Q_UNIT_HINT)
    print("raw loaded shape:", cube.data.shape, "(ndim", cube.ndim, ")")
    if cube.ndim<3:
        raise ValueError(f"{cube.ndim}D — not a scan; 로더가 데이터셋 검색 후에도 2D면 직접 로드하세요.")
    if DET_BIN>1: cube=fds.bin_cube_detector(cube, DET_BIN)
scan=cube.scan_shape; dp=cube.dp_shape
QPP = cube.calibration.q_per_px or Q_PER_PX
DR  = fds.quefrency_per_px(dp[0], QPP)      # 켑스트럼 픽셀당 Å
print("cube:", cube.shape, "| scan:", scan, "| dp:", dp)
print(f"q_per_px = {QPP:.5g} 1/A/px  ->  cepstral dr = {DR:.4g} A/px,  r_max ~ {DR*(dp[0]//2):.1f} A")

# 표시용 맵 재배열(3D 스택 대비) + 저장 헬퍼
import math
def _mapshape(n):
    r=int(math.sqrt(n))
    while r>1 and n%r: r-=1
    return (r,n//r) if r>1 else (1,n)
MAP=tuple(scan) if len(scan)==2 else _mapshape(int(np.prod(scan)))
def as_map(v):
    v=np.asarray(v,float).ravel(); m=np.full(int(np.prod(MAP)),np.nan); m[:min(v.size,m.size)]=v[:m.size]; return m.reshape(MAP)
SAVE_DIR=(os.path.dirname(DM4_PATH)+"/nb6_outputs" if not USE_SYNTHETIC else "nb6_outputs")
os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,name): p=os.path.join(SAVE_DIR,name+".png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(name,header,rows):
    import csv; p=os.path.join(SAVE_DIR,name+".csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(header); w.writerows(rows)
    print("saved:",p)
print("outputs ->", os.path.abspath(SAVE_DIR))



## 2) median · **MAX 프로젝션** · 물질 마스크

median은 약해서 잘 안 보일 때, **MAX 프로젝션**(각 검출기 픽셀의 스캔 전체 최댓값)이 **가장 강한 신호**
(결정 스팟 등)를 모아 보여줍니다. 그리고 산란 세기로 **물질 마스크**를 만들어 **빈(진공) 위치를 분석에서
제외**합니다(흰색=분석). 이후 모든 단계는 물질 위치만 씁니다.


In [ ]:

med=fds.median_pattern(cube); mx=cube.max_dp()
if CENTER is not None: cx,cy=CENTER
else: cx,cy=fds.center_of_mass(med, threshold=0.3)

# 물질 마스크 (빈 영역 제외)
if MATERIAL_MASK and len(scan)==2:
    empty=None
    if EMPTY_ROI is not None:
        empty=np.zeros(scan,bool); y0,y1,x0,x1=EMPTY_ROI; empty[y0:y1,x0:x1]=True
    elif EMPTY_ROWS:
        empty=np.zeros(scan,bool); empty[max(0,scan[0]-EMPTY_ROWS):,:]=True
    material=fds.material_mask(cube, center=(cx,cy), empty_mask=empty)
    if ERODE_EDGE > 0:                               # 얇은 가장자리(두께 효과) 제외 → 벌크만
        try:
            from scipy.ndimage import binary_erosion
            material = binary_erosion(material, iterations=int(ERODE_EDGE))
        except Exception: pass
else:
    material=np.ones(scan,bool)
KEEP=material.ravel()
def scatter(vec, fill=np.nan):   # 물질 전용 결과를 스캔 전체로 되돌림(빈 곳=fill)
    out=np.full(KEEP.size,fill,float); out[KEEP]=np.asarray(vec,float).ravel(); return out.reshape(scan)
print(f"center: ({cx:.1f},{cy:.1f}) | material positions: {int(KEEP.sum())}/{KEEP.size} ({100*KEEP.mean():.0f}%)")

fig,ax=plt.subplots(1,3,figsize=(13,4.2))
im=ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(cx,cy,"c+",ms=10); ax[0].set_title("median (log)"); ax[0].axis("off")
im=ax[1].imshow(np.log1p(mx),cmap="magma"); ax[1].plot(cx,cy,"c+",ms=10); ax[1].set_title("MAX projection (strongest signal)"); ax[1].axis("off")
ax[2].imshow(material,cmap="gray"); ax[2].set_title("material mask (white = analyzed)"); ax[2].axis("off")
plt.tight_layout(); save(fig,"02_median_max_material"); plt.show()
np.save(os.path.join(SAVE_DIR,"02_max_projection.npy"), np.asarray(mx))



## 2b) 빔 wander 점검 (전처리)

넓은 영역에서 **직접빔이 흔들리거나 기울면**, 원시 패턴 PCA는 구조 대신 빔 움직임을 잡습니다(nb5 §4에서
확인됨). 각 위치의 **빔 중심 이동**을 매핑해 그 정도를 봅니다. 켑스트럼(§3~5)은 **병진 불변**이라 영향
없지만, **영역별 RDF(§6)** 는 민감하므로 §6에서 각 패턴을 **자기 빔 중심으로 정렬 후 평균**합니다.


In [ ]:

comx, comy = fds.center_of_mass_map(cube, normalize=True)   # 위치별 빔 이동(기하중심 대비)
comx = np.asarray(comx); comy = np.asarray(comy)
shift = np.hypot(comx, comy)
print(f"beam wander: median {np.nanmedian(shift[material]):.2f} px, "
      f"max {np.nanmax(shift[material]):.2f} px (물질 영역)")
fig,ax=plt.subplots(1,3,figsize=(14,3.6))
for a,d,t in [(ax[0],np.where(material,comx,np.nan),"beam shift x (px)"),
              (ax[1],np.where(material,comy,np.nan),"beam shift y (px)"),
              (ax[2],np.where(material,shift,np.nan),"|beam shift| (px)")]:
    im=a.imshow(d,cmap="coolwarm" if "shift x" in t or "shift y" in t else "viridis")
    a.set_title(t); a.axis("off"); plt.colorbar(im,ax=a,fraction=0.046)
fig.suptitle("beam wander across the scan (smooth gradient = beam tilt)", y=1.04)
plt.tight_layout(); save(fig,"02b_beam_wander"); plt.show()


## 2c) 노이즈(고정 hot/dead 픽셀) · **캘리브레이션** · **중심빔 제거 점검**

**노이즈**: 매 프레임 같은 위치의 hot/dead 픽셀은 평균으로도 안 없어지고 log(EWPC)를 왜곡합니다. bad-pixel
맵으로 검출하고, 영역 평균 패턴은 §6 RDF 전에 정리합니다. `DENOISE_CUBE=True`면 전체 큐브 수리.

**캘리브레이션**: 비정질 첫 링은 **FSDP**. 결정 격자간격 $d=1/q$ 와 **최근접이웃** $r_{nn}\approx1.23/q$
(Ehrenfest)를 구분하세요 — **RDF 첫 peak = $1.23/q$**. peak이 1.5 Å면 링이 $q\approx0.82$로 잡힌 것
→ q 스케일 ~30% 큼. `CALIB_R_TARGET`(예 2.0 Å)로 보정.

**중심빔 제거 점검**: I(q)에서 **직접빔 끝(valley) `q_beam`** 을 추정해, RDF 빔컷 `q_int_min`(파랑 -·-)이
그 바깥인지 확인합니다(주황 음영=직접빔 영역). 부족하면 `q_int_min`을 올리세요. 또한 **cepstrum**은 빔을
따로 안 빼지만(log가 세기를 압축) NBED **디스크 경계**가 `r≈1/q_beam`에 인공물을 남길 수 있어, §3에 그
위치를 표시합니다(그 근처 피크는 원자신호가 아닐 수 있음).

In [ ]:
badmap = fds.bad_pixel_map(np.asarray(cube.max_dp(),float), hot_threshold=HOT_THRESHOLD)
print(f"bad (hot/dead) pixels: {int(badmap.sum())} ({100*badmap.mean():.2f}% of detector)")
if DENOISE_CUBE and badmap.sum()>0:
    cube = fds.repair_bad_pixels(cube, badmap); print('repaired whole cube')

# --- 평균 패턴은 '물질 위치만' (진공 제외) ---
mp = fds.average_pattern(cube, material)
print(f'calibration pattern = mean over {int(material.sum())} MATERIAL positions '
      f'(vacuum excluded: {int((~material).sum())}); MATERIAL_MASK={MATERIAL_MASK}')
qd, Id = fds.azimuthal_integrate(mp, (cx,cy), q_per_px=QPP)

# --- 빔 배경/잔차 먼저 → 빔봉우리와 첫 링 사이 '골짜기'를 빔 끝으로(10%-crossing은 꼬리까지 잡아 과대추정) ---
_,__,fsdp_base,fsdp_res = fds.find_fsdp(qd, Id, q_lo=0.10, q_hi=0.95, return_curves=True)
bp=int(np.argmax(np.where(qd<0.15, fsdp_res, -np.inf)))
hi=int(np.searchsorted(qd,0.45))
q_beam=float(qd[bp+int(np.argmin(fsdp_res[bp:hi]))]) if hi>bp+1 else float(CFG_RDF.q_int_min)
r_beam=(1.0/q_beam) if q_beam>0 else np.inf

# --- 첫 링(FSDP): 빔 감소 배경 제거 후 물리적 창에서 검출 + 신뢰도 ---
q_ring, ring_conf = fds.find_fsdp(qd, Id, q_lo=max(q_beam,0.12), q_hi=0.95)

# --- 캘리브레이션 체인(투명) ---
qpp_unbin = QPP/max(DET_BIN,1)
print('\n[calibration chain]')
print(f'  dm 메타데이터 ~ {qpp_unbin*10:.5f} 1/nm/px (unbinned)  [주신 0.043888과 일치?]')
print(f'  -> /10 = {qpp_unbin:.6f} 1/A/px  x DET_BIN({DET_BIN}) = QPP {QPP:.6f} 1/A/px  ← 사용값')
print(f'  FSDP: q_ring={q_ring:.3f} 1/A (confidence {ring_conf:.1f}) | d=1/q={1/q_ring:.2f} | nn~1.23/q={1.23/q_ring:.2f} A')
if ring_conf < 5:
    print(f'  \u26a0 FSDP 신뢰도 {ring_conf:.1f} < 5 \u2192 뚜렷한 링 없음(약한 신호). q_ring/nn 믿지 말고 메타데이터 calibration을 신뢰.')
else:
    print(f'  \u2713 링 검출 신뢰 가능(conf {ring_conf:.1f}).')

if CALIB_R_TARGET and np.isfinite(q_ring) and q_ring>0 and ring_conf>=5:
    q_target=1.23/CALIB_R_TARGET; scale=q_target/q_ring
    QPP=QPP*scale; DR=fds.quefrency_per_px(dp[0],QPP); cube.calibration.q_per_px=QPP
    qd,Id=fds.azimuthal_integrate(mp,(cx,cy),q_per_px=QPP); q_ring=q_target
    print(f'  [!] CALIB_R_TARGET={CALIB_R_TARGET} 강제 -> QPP x{scale:.3f} = {QPP:.6f} (메타 {abs(1-scale)*100:.0f}% 보정)')
elif CALIB_R_TARGET and ring_conf<5:
    print('  [!] CALIB_R_TARGET 설정됐지만 링 신뢰도 낮아 강제보정 건너뜀(메타데이터 유지).')
else:
    print('  CALIB_R_TARGET=None -> 메타데이터 그대로 신뢰.')
print(f'  direct-beam edge q_beam={q_beam:.3f} | cepstrum 빔인공물 r~{r_beam:.2f} A')

# --- 그림: bad-pixel | I(q)+컷 | 중심 확대(crosshair) ---
import matplotlib.patches as mpatches
fig,ax=plt.subplots(1,3,figsize=(15,4.3))
ax[0].imshow(badmap,cmap='gray'); ax[0].set_title(f'bad-pixel map ({int(badmap.sum())})'); ax[0].axis('off')
ax[1].plot(qd,Id,'k-',label='I(q)'); ax[1].plot(qd,fsdp_base,'g--',lw=1,label='beam baseline')
ax[1].axvspan(0,q_beam,color='orange',alpha=0.13,label=f'beam(q<{q_beam:.2f})')
ax[1].axvline(CFG_RDF.q_int_min,color='b',ls='-.',lw=1,label=f'RDF cut={CFG_RDF.q_int_min}')
if np.isfinite(q_ring): ax[1].axvline(q_ring,color='r',ls='--',label=f'FSDP q={q_ring:.2f} (conf {ring_conf:.0f})')
axr=ax[1].twinx(); axr.plot(qd,fsdp_res,color='0.6',lw=0.8); axr.axhline(0,color='0.8',lw=0.5)
axr.set_ylabel('residual (I - baseline)', color='0.5', fontsize=8)
ax[1].set_xlabel('q (1/A)'); ax[1].set_ylabel('I(q)'); ax[1].set_title('mean I(q): ring = bump in gray residual?'); ax[1].legend(fontsize=6, loc='upper right')
# 중심 확대(고정 ±ZOOM, crosshair로 정확히 확인)
ZOOM=int(min(CENTER_ZOOM, dp[0]//2, dp[1]//2))
y0,y1=max(0,int(round(cy))-ZOOM),min(dp[0],int(round(cy))+ZOOM); x0,x1=max(0,int(round(cx))-ZOOM),min(dp[1],int(round(cx))+ZOOM)
ax[2].imshow(np.log1p(mp[y0:y1,x0:x1]),cmap='magma',extent=[x0,x1,y1,y0])
ax[2].axhline(cy,color='cyan',lw=0.7,ls='--'); ax[2].axvline(cx,color='cyan',lw=0.7,ls='--')
ax[2].plot(cx,cy,'c+',ms=18,mew=2)
ax[2].set_title(f'center zoom (+/-{ZOOM}px): + on beam?'); ax[2].set_xlabel('det x (px)'); ax[2].set_ylabel('det y (px)')
plt.tight_layout(); save(fig,'02c_calibration_center'); plt.show()


## 2d) 강한 신호 링 추출 시도 — 상위 산란 위치만 정렬평균 + 빔배경 제거

평균이 약한 위치까지 섞으면 약한 링이 더 묻힙니다. **빔 밖(q 0.3~0.95) 산란이 가장 강한 상위
`STRONG_FRAC` 위치만** 골라 **빔중심 정렬 평균** 후, find_fsdp의 빔 배경을 제거해 **희미한 FSDP가
드러나는지** 봅니다. residual(회색)에 bump가 서고 confidence가 5 이상이면 거기서 거리/화합물 비교가
가능합니다. 그래도 없으면 이 데이터로는 절대거리 판별이 어렵다는 최종 확인입니다.

In [ ]:
from scipy.signal import find_peaks
ri_px, ro_px = 0.30/QPP, 0.95/QPP
adf = np.asarray(fds.annular_dark_field(cube, (cx,cy), ri_px, ro_px), float)
thr = np.nanpercentile(np.where(material, adf, np.nan), 100*(1-STRONG_FRAC))
strong = material & (adf >= thr)
print(f'strong-signal positions: {int(strong.sum())}/{int(material.sum())} (top {STRONG_FRAC*100:.0f}%)')
tgt=(dp[1]/2.0, dp[0]/2.0)
pat_s = fds.clean_pattern(fds.average_pattern_aligned(cube, strong, target=tgt, threshold=0.3), hot_threshold=HOT_THRESHOLD)
qs, Is = fds.azimuthal_integrate(pat_s, tgt, q_per_px=QPP)
# 빔 배경 baseline + 잔차 (넓은 링 보존 위해 낮은 창)
_,__ ,base_s,res_s = fds.find_fsdp(qs, Is, q_lo=0.10, q_hi=0.95, return_curves=True)
# 빔봉우리(q<0.15) 다음의 '골짜기'를 찾아 링 검출 시작점 q_lo로 (빔과 링 분리)
bp=int(np.argmax(np.where(qs<0.15, res_s, -np.inf)))
hi=int(np.searchsorted(qs,0.45))
vrel=int(np.argmin(res_s[bp:hi])) if hi>bp+1 else 0
q_valley=float(qs[bp+vrel]); q_lo_use=max(q_valley, 0.12)
print(f'beam-ring valley at q={q_valley:.3f} -> ring 검출 창 [{q_lo_use:.2f}, 0.95]')
# 잔차에서 후보 링 '여러 개' 나열(약하고 넓은 링도)
mwin=(qs>=q_lo_use)&(qs<=0.95); qw,rw=qs[mwin],res_s[mwin]
noise=np.median(np.abs(rw-np.median(rw)))*1.4826+1e-9
pk,props=find_peaks(rw, prominence=0.8*noise)
order=np.argsort(props['prominences'])[::-1]
print('candidate rings (residual peaks): q | d=1/q | nn~1.23/q | prominence/noise')
cand=[]
for i in order[:5]:
    qi=float(qw[pk[i]]); c=float(props['prominences'][i]/noise); cand.append((qi,c))
    print(f'   q={qi:.3f} | d={1/qi:.2f} | nn={1.23/qi:.2f} A | conf {c:.1f}')
if RING_Q_MANUAL:
    q_use=float(RING_Q_MANUAL); print(f'[manual] RING_Q_MANUAL={q_use} 사용 -> d={1/q_use:.2f} nn={1.23/q_use:.2f} A')
elif cand:
    q_use=cand[0][0]; print(f'[auto] 최강 후보 q={q_use:.3f} (conf {cand[0][1]:.1f}) 사용. 감마 이미지의 링과 다르면 RING_Q_MANUAL로 지정하세요.')
else:
    q_use=np.nan; print('후보 링 없음 — 감마 이미지에서 링이 보이면 그 q를 RING_Q_MANUAL에 넣으세요.')

fig,ax=plt.subplots(1,3,figsize=(15,4.3))
# (1) 감마 강조 2D (사용자가 링 본 방식 재현)
g=np.clip(pat_s,0,None); g=(g/g.max())**GAMMA
ax[0].imshow(g,cmap='gray'); ax[0].plot(tgt[0],tgt[1],'c+',ms=8)
if np.isfinite(q_use): ax[0].add_patch(__import__('matplotlib').patches.Circle(tgt,q_use/QPP,fill=False,ec='cyan',ls='--',lw=1))
ax[0].set_title(f'strong mean, gamma={GAMMA} (faint rings)'); ax[0].axis('off')
# (2) I(q) 로그 y — 약한 링이 shoulder로 보임
ax[1].semilogy(qs,np.clip(Is,1e-3,None),'k-'); ax[1].semilogy(qs,np.clip(base_s,1e-3,None),'g--',lw=1,label='beam baseline')
for qi,c in cand: ax[1].axvline(qi,color='r',ls=':',lw=0.8)
ax[1].set_xlabel('q (1/A)'); ax[1].set_ylabel('I(q) [log]'); ax[1].set_title('log I(q): rings = shoulders'); ax[1].legend(fontsize=7)
# (3) residual — 링 bump
ax[2].plot(qs,res_s,'-',color='0.4'); ax[2].axhline(0,color='0.8',lw=0.6); ax[2].axvspan(0,q_lo_use,color='orange',alpha=0.1)
for qi,c in cand: ax[2].axvline(qi,color='r',ls='--',lw=0.8); ax[2].text(qi,ax[2].get_ylim()[1]*0.9,f'{qi:.2f}',fontsize=6,color='r',ha='center')
ax[2].set_xlabel('q (1/A)'); ax[2].set_ylabel('residual (I-baseline)'); ax[2].set_title('ring bumps (candidates marked)')
plt.tight_layout(); save(fig,'02d_strong_signal_fsdp'); plt.show()


## 2e) 후보 화합물과 **링-패턴 매칭** — 어떤 Li 화합물인가

약하고 넓은 링엔 RDF보다 **링 q 위치를 화합물 d-spacing과 직접 대조**하는 게 맞습니다. §2d 측정 링 q를
각 후보의 주요 회절 링(결정 d→q)과 비교해 점수를 매깁니다. **LiF/Li2O/Li2S는 저-q(d~4Å) 링이 없어**
이 측정 링과 안 맞고, **큰 단위셀(Li2CO3/Li3N)** 만 후보가 됩니다. 링이 1~2개뿐이면 근거는 약하니, 강한
링이 더 잡히면(더 긴 노출) 신뢰가 올라갑니다.

In [ ]:
meas_q = [c[0] for c in cand] if cand else []
if RING_Q_MANUAL: meas_q = [float(RING_Q_MANUAL)] + meas_q
ranked = fds.match_rings(meas_q, tol=0.03)
print('measured rings (1/A):', [round(q,3) for q in meas_q], '| d(A):', [round(1/q,2) for q in meas_q if q>0])
print('compound ring-pattern match (score = matched-ring intensity fraction):')
for c,score,matched,missing,mindq in ranked:
    print(f'  {c:7s} score {score:.2f} | matched q {matched} | \u0394q_min {mindq} | missing-strong q {missing}')
if ranked and ranked[0][1]>0:
    closest=min(ranked,key=lambda x:x[4])
    print(f'\n\u2192 best score: {ranked[0][0]} ({ranked[0][1]:.2f}) | 위치 최근접: {closest[0]} (\u0394q={closest[4]}).')
    print('  링 1개면 잠정적 — LiF/Li2O/Li2S 배제, 큰-cell(Li2CO3/Li3N)만 후보. 강한 링/EELS로 확정 권장.')
else:
    print('\n측정 링이 후보 화합물 링과 안 맞음 — RING_Q_MANUAL 확인 또는 데이터 한계.')

colors={'LiF':'#e41a1c','Li2O':'#377eb8','Li3N':'#4daf4a','Li2CO3':'#984ea3','Li2S':'#ff7f00'}
fig,ax=plt.subplots(1,1,figsize=(11,4.6))
ax.plot(qs, res_s, 'k-', lw=1.3, label='measured residual (strong signal)'); ax.axhline(0,color='0.85',lw=.6)
for c in fds.COMPOUND_RINGS:
    for q_c,w in fds.compound_ring_q(c):
        if q_c<=qs.max(): ax.axvline(q_c,color=colors[c],ls='--',lw=0.4+1.0*w,alpha=0.65)
    ax.plot([],[],color=colors[c],lw=2,label=f'{c} rings')
for q in meas_q:
    if q>0: ax.axvline(q,color='k',ls=':',lw=1.4); ax.text(q,ax.get_ylim()[1]*0.9,f'{q:.2f}',fontsize=7,ha='center')
ax.set_xlim(0,1.1); ax.set_xlabel('q (1/A)'); ax.set_ylabel('residual (I - beam)'); ax.set_title('measured ring(s) vs candidate compound ring patterns'); ax.legend(fontsize=8,ncol=3)
plt.tight_layout(); save(fig,'02e_ring_match'); plt.show()
save_csv('02e_ring_match',['compound','score','matched_q','missing_strong_q','min_dq'],
         [[c,f'{s:.3f}',str(m),str(mi),dq] for c,s,m,mi,dq in ranked])


## 2f) 측정 그래프 vs **5개 화합물 그래프** — 유사도 + 공간 분포

각 후보의 **기대 회절 그래프**(링 d→q, 강도 가중)를 합성해 측정 그래프(강한신호 residual)와 나란히 놓고
**유사도(코사인)** 로 순위를 매깁니다. 그리고 측정 링(q_use)에서 **환형 가상이미지**를 만들어 이 상이 스캔
**어디에 분포**하는지 봅니다. (링 1개라 유사도는 대체로 낮게 나오며, 링이 가장 가까운 화합물이 최상위.)

In [ ]:
qref = qs
meas = np.clip(res_s,0,None); meas = meas/(np.linalg.norm(meas)+1e-12)
# 순위는 §2e와 동일한 '측정 링 위치 최근접'(match_rings) 기준으로 통일 — 코사인은 구역 겹침이라 오해 소지.
rk = fds.match_rings(meas_q, tol=0.05)
sims = [(c, score, mindq) for c,score,matched,missing,mindq in rk]
print('compound match by measured-ring POSITION (Δq = 링 위치 차이, 작을수록 일치):')
for c,score,mindq in sims: print(f'  {c:7s} score {score:.2f} | Δq {mindq}')
top=[c for c,_,dq in sims if dq<=0.03]
excl=[c for c,_,dq in sims if dq>0.05]
print(f'\n\u2192 측정 링(d={1/q_use:.2f}A) 일치 후보: {top} | 배제(이 링 없음): {excl}')
print('  주의: d~4.1A는 Li2CO3(4.16)와 Li3N(3.87) 사이 \u2014 링 1개로는 둘 구별 불가. 확정엔 EELS 필요.')

fig,ax=plt.subplots(1,5,figsize=(20,3.4),sharey=True)
for i,(c,score,mindq) in enumerate(sims):
    ref=fds.synth_compound_iq(c,qref,0.04)
    ax[i].plot(qref,meas/meas.max(),'k-',lw=1.3,label='measured')
    ax[i].plot(qref,ref/ref.max(),color=colors[c],lw=1.5,label=c)
    ax[i].axvline(q_use,color='0.6',ls=':',lw=0.8) if np.isfinite(q_use) else None
    ax[i].set_xlim(0,1.0); ax[i].set_title(f'{c}  \u0394q={mindq}',fontsize=10); ax[i].set_xlabel('q (1/A)'); ax[i].legend(fontsize=7)
ax[0].set_ylabel('normalized')
plt.tight_layout(); save(fig,'02f_compound_graphs'); plt.show()

# 공간 분포: 측정 링에서 환형 가상이미지 -> 이 상이 어디에
if np.isfinite(q_use):
    rw=0.04
    dfmap=np.asarray(fds.annular_dark_field(cube,(cx,cy),(q_use-rw)/QPP,(q_use+rw)/QPP),float)
    fig2,ax2=plt.subplots(1,2,figsize=(11,3.8))
    im=ax2[0].imshow(np.where(material,dfmap,np.nan),cmap='inferno'); ax2[0].set_title(f'ring DF @ q={q_use:.2f} (d={1/q_use:.1f}A) = phase map'); ax2[0].axis('off'); plt.colorbar(im,ax=ax2[0],fraction=0.046)
    ax2[1].hist(dfmap[material],bins=40,color='0.4'); ax2[1].set_title('ring intensity histogram (material px)'); ax2[1].set_xlabel('ring DF intensity')
    plt.tight_layout(); save(fig2,'02f_ring_map'); plt.show()
    save_csv('02f_ring_position_match',['compound','score','dq_min'],[[c,f'{sc:.3f}',dq] for c,sc,dq in sims])


## 2g) MAX 프로젝션의 **결정질 링** 분석 — SEI 결정상 찾기

비정질 평균 I(q)엔 넓은 halo만 보이지만, **MAX 프로젝션**(각 검출기 픽셀의 스캔 전체 최댓값)은 스캔 곳곳의
**결정 알갱이 Bragg 스팟을 모두 모아** powder-like 링으로 만듭니다. 이 링들은 **날카롭고 여러 개** → 결정질
화합물의 d-spacing과 직접 매칭됩니다. 결정질 LiF/Li2O/Li2CO3 등이 있으면 여기서 드러납니다.

In [ ]:
from scipy.signal import find_peaks
mxp = fds.clean_pattern(np.asarray(cube.max_dp(),float), hot_threshold=HOT_THRESHOLD)  # MAX는 hot pixel 증폭 → 정리
qm, Im = fds.azimuthal_integrate(mxp, (cx,cy), q_per_px=QPP)
_,__,base_m,res_m = fds.find_fsdp(qm, Im, q_lo=0.10, q_hi=float(qm.max()), return_curves=True)
mwin=(qm>=max(q_beam,0.15))&(qm<=qm.max())
qw,rw=qm[mwin],res_m[mwin]
noise=np.median(np.abs(rw-np.median(rw)))*1.4826+1e-9
pk,props=find_peaks(rw, prominence=1.2*noise, distance=3)
order=np.argsort(props['prominences'])[::-1]
ring_qs=[float(qw[pk[i]]) for i in order[:8]]
print('MAX 결정 링 후보 (q | d=1/q | conf):')
for i in order[:8]:
    qi=float(qw[pk[i]]); print(f'   q={qi:.3f} | d={1/qi:.2f} A | conf {props["prominences"][i]/noise:.1f}')
RING_TBL = fds.ALL_RINGS if INCLUDE_SUBSTRATE else fds.COMPOUND_RINGS   # Cu 없으면 Li 화합물만
ranked = fds.match_rings(ring_qs, rings=RING_TBL, tol=0.025)
# 각 측정 링을 어떤 상이 설명하나(가장 가까운 것들)
print('per-ring assignment (측정 링 -> 가능한 상):')
for q in ring_qs:
    hits=[f"{c}(d={d})" for c,rr in RING_TBL.items() for d,w in rr if abs(1/d-q)<=0.025]
    print(f'   q={q:.3f} (d={1/q:.2f} A): {hits}')
print('\ncrystalline phase match (여러 링이 맞을수록 score↑):')
for c,score,matched,missing,mindq in ranked:
    print(f'  {c:7s} score {score:.2f} | matched q {matched} | missing-strong q {missing}')
print('\n\u2192 여러 상이 섞였을 수 있음. d>3.3 링은 Li 화합물(Cu 불가), d~1.5는 Cu2O/CuO(구리산화물).')
# EDS: O,C 많고 N,F,S 있음 → 5개 Li상 모두 화학적 가능. 각 상의 링 위치에 실제 세기가 있나 직접 탐침(약한/비정질도 포착).
print('\nring-position presence probe (각 상의 링 위치에 MAX 세기; peak 아니어도 보임):')
probe_list = list(fds.ALL_RINGS) if INCLUDE_SUBSTRATE else list(fds.COMPOUND_RINGS)
for c in probe_list:
    hits=[]
    for d,w in fds.ALL_RINGS[c]:
        qc=1.0/d
        if q_beam<=qc<=qm.max():
            win=np.abs(qm-qc)<=0.02
            hits.append((round(qc,3), round(float(np.max(res_m[win])/noise),1) if win.any() else 0.0))
    strong=[h for h in hits if h[1]>=2.0]
    print(f'  {c:7s}: present(conf>=2) {strong}  |  all {hits}')
print('  \u2192 위 per-ring assignment로 상별 링을 확인하세요(단일 화합물로 단정 금지).')

colors={'LiF':'#e41a1c','Li2O':'#377eb8','Li3N':'#4daf4a','Li2CO3':'#984ea3','Li2S':'#ff7f00',
        'Cu':'#8c564b','Cu2O':'#17becf','CuO':'#7f7f7f','Li':'#e377c2'}
fig,ax=plt.subplots(1,3,figsize=(16,4.3))
g=np.clip(mxp,0,None); g=(g/g.max())**0.3
ax[0].imshow(g,cmap='gray'); ax[0].plot(cx,cy,'c+',ms=8); ax[0].set_title('MAX projection (gamma) — spots/rings?'); ax[0].axis('off')
ax[1].semilogy(qm,np.clip(Im,1e-2,None),'k-'); ax[1].semilogy(qm,np.clip(base_m,1e-2,None),'g--',lw=1,label='baseline')
for qi in ring_qs: ax[1].axvline(qi,color='r',ls=':',lw=0.8)
ax[1].set_xlabel('q (1/A)'); ax[1].set_ylabel('MAX I(q) [log]'); ax[1].set_title('MAX radial: sharp crystalline rings'); ax[1].legend(fontsize=7)
ax[2].plot(qm,res_m,'-',color='0.4')
for c in RING_TBL:
    for q_c,w in fds.compound_ring_q(c,rings=RING_TBL):
        if q_c<=qm.max(): ax[2].axvline(q_c,color=colors.get(c,'0.5'),ls='--',lw=0.3+0.9*w,alpha=0.6)
    ax[2].plot([],[],color=colors.get(c,'0.5'),lw=2,label=c)
for qi in ring_qs: ax[2].axvline(qi,color='k',ls=':',lw=1.0)
ax[2].set_xlim(0,qm.max()); ax[2].set_xlabel('q (1/A)'); ax[2].set_ylabel('MAX residual'); ax[2].set_title('MAX rings vs Li + substrate patterns'); ax[2].legend(fontsize=6,ncol=3)
plt.tight_layout(); save(fig,'02g_max_crystalline'); plt.show()
save_csv('02g_max_rings',['q_invA','d_A'],[[f'{q:.4f}',f'{1/q:.3f}'] for q in ring_qs])


## 2h) 결정질 알갱이 위치 지도 — 결정 링에서 가상 암시야(DF)

§2g에서 찾은 **결정 링**(d-spacing)에서 환형 가상 암시야 이미지를 만들면, 그 링을 내는 **결정 알갱이가
스캔의 어디에 있는지** 밝게 보입니다. 여러 결정 링의 DF를 합치면 결정질 상의 공간 분포가 나옵니다.

In [ ]:
# §2g의 ring_qs(결정 링) 각각에서 환형 DF → 결정 알갱이 위치
rings_use = [q for q in ring_qs if q>max(q_beam,0.18)][:4]
rw=0.035
dfs=[np.asarray(fds.annular_dark_field(cube,(cx,cy),(q-rw)/QPP,(q+rw)/QPP),float) for q in rings_use]
nrow=len(dfs)
fig,ax=plt.subplots(1,nrow+1,figsize=(3.4*(nrow+1),3.6))
for j,(q,d) in enumerate(zip(rings_use,dfs)):
    im=ax[j].imshow(np.where(material,d,np.nan),cmap='inferno'); ax[j].set_title(f'DF @ q={q:.2f} (d={1/q:.2f}A)',fontsize=9); ax[j].axis('off'); plt.colorbar(im,ax=ax[j],fraction=0.046)
# 결정질 종합: 여러 링 DF 합(정규화) — 밝음=결정 알갱이
cryst=np.zeros_like(dfs[0])
for d in dfs:
    dd=np.where(material,d,np.nan); cryst=cryst+ (dd-np.nanmin(dd))/(np.nanmax(dd)-np.nanmin(dd)+1e-9)
im=ax[nrow].imshow(cryst,cmap='viridis'); ax[nrow].set_title('crystalline sum (bright=grains)',fontsize=9); ax[nrow].axis('off'); plt.colorbar(im,ax=ax[nrow],fraction=0.046)
plt.tight_layout(); save(fig,'02h_crystalline_grains'); plt.show()
frac_cryst = float(np.nanmean(cryst[material]>np.nanpercentile(cryst[material],75)))
print(f'결정 링 DF가 강한 상위 위치 비율(대략적 결정질 분포): {frac_cryst*100:.0f}% (상위25% 기준)')
save_csv('02h_crystalline_map',['scan_index','crystalline_sum'],[[i,f'{cryst.ravel()[i]:.4f}'] for i in range(cryst.size)])


## 2i) 링 패턴 NNLS 언믹싱 — EDS 확정 Li 상 함량

EDS로 원소(O,C 많고 N,F,S)가 확정됐으니, 측정 회절 링 패턴(MAX residual)을 **5개 Li 화합물의 링 패턴**
선형결합으로 분해합니다: `측정 ≈ Σ aᵪ·(화합물ᵪ 링패턴)`, aᵪ≥0. 없는 상은 0. 기판 Cu 없음 → Li 상만.

In [ ]:
from scipy.optimize import nnls
CANDIDATES = ['Li2CO3','LiF','Li3N','Li2S','Li2O']   # EDS: C,O,N,F,S. 기판 Cu 없음
qfit=(qm>=max(q_beam,0.18))&(qm<=1.05)
Rmat=np.vstack([fds.synth_compound_iq(c,qm,sigma_q=SIG_Q,rings=fds.COMPOUND_RINGS) for c in CANDIDATES]).T
target=np.clip(res_m,0,None)
a,rn=nnls(Rmat[qfit], target[qfit])
frac=a/(a.sum()+1e-12)
recon=Rmat@a
print('MAX 링 패턴 NNLS 언믹싱 (Li 상 함량):')
for c,f in sorted(zip(CANDIDATES,frac),key=lambda x:-x[1]): print(f'  {c:7s} {f*100:5.1f}%')
fitres=rn/(np.linalg.norm(target[qfit])+1e-12)
print(f'fit residual(작을수록 좋음): {fitres:.3f}')
if fitres>0.5:
    print('  \u26a0 fit 나쁨(>0.5) \u2192 링이 겹치고 강도표가 근사라 정량 함량 신뢰 불가.')
    print('     \u2192 상 존재 여부는 §2g presence probe(각 상 링 위치 세기)를 기준으로 판단하세요.')
    # 진단: 각 상의 고유(최저-q) 링에 세기가 있는데 함량 0이면 존재하나 unmix가 놓친 것
    for c in CANDIDATES:
        qd0=min(1.0/d for d,_ in fds.COMPOUND_RINGS[c]); w0=np.abs(qm-qd0)<=0.025
        if w0.any() and float(np.max(res_m[w0])/noise)>=2 and frac[CANDIDATES.index(c)]<0.05:
            print(f'     \u2022 {c}: 고유링 q={qd0:.3f}(d={1/qd0:.2f}) 세기 있음(존재) but unmix 0% \u2192 존재로 간주')

fig,ax=plt.subplots(1,2,figsize=(13,4.3))
ax[0].plot(qm,target,'k-',lw=1.6,label='measured (MAX residual)')
ax[0].plot(qm,recon,'r--',lw=1.3,label='NNLS reconstruction')
for c,ai in zip(CANDIDATES,a):
    if ai>1e-9: ax[0].plot(qm, ai*fds.synth_compound_iq(c,qm,SIG_Q,rings=fds.COMPOUND_RINGS), color=colors[c], lw=1.0, alpha=0.8, label=f'{c} {ai/(a.sum()+1e-12)*100:.0f}%')
ax[0].set_xlim(0,1.05); ax[0].set_xlabel('q (1/A)'); ax[0].set_ylabel('residual'); ax[0].set_title('ring-pattern unmix (EDS Li phases)'); ax[0].legend(fontsize=7)
ax[1].bar(range(len(CANDIDATES)),[frac[i]*100 for i in range(len(CANDIDATES))],color=[colors[c] for c in CANDIDATES])
ax[1].set_xticks(range(len(CANDIDATES))); ax[1].set_xticklabels(CANDIDATES,rotation=30); ax[1].set_ylabel('abundance %'); ax[1].set_title('phase fraction (ring-domain)')
plt.tight_layout(); save(fig,'02i_ring_unmix'); plt.show()
save_csv('02i_ring_unmix',['compound','fraction_pct'],[[c,f'{frac[i]*100:.1f}'] for i,c in enumerate(CANDIDATES)])


## 3) 평균 EWPC + 켑스트럼 방사 프로파일 (원자간 거리)

전체 평균 켑스트럼과 그 방사 프로파일. **피크 = 원자간 거리(Å)**. 이게 약한 신호에서도 나오는 게 핵심.
색 밴드는 §4 FC-STEM 거리 구간. **주황 -·- 선 = 중심빔 디스크 경계 인공물 위치(≈1/q_beam, §2c)** — 그
근처 피크는 원자신호가 아닐 수 있으니 주의. 피크가 물리적으로 타당한 Å(예 2~4)인지 확인하세요.

In [ ]:
matflat = cube._flat_patterns()[KEEP]          # 물질 위치 패턴만
mcep=fds.ewpc_mean(matflat, offset=EWPC_OFFSET, window=WINDOW, reducer="mean", n_jobs=N_JOBS, progress=True)
r_ax, prof=fds.cepstral_radial_profile(mcep, QPP, r_min=0.4)
from scipy.signal import find_peaks as _fp
_ms=r_ax<6.0; _pk,_=_fp(prof[_ms], prominence=0.05*np.ptp(prof[_ms])) if _ms.sum()>3 else ([],None)
if len(_pk)==0:
    print(f"\u26a0 평균 cepstral 방사 프로파일에 뚜렷한 peak 없음(단조 감소). cepstral dr={DR:.2f} A는 검출기 전체 q-범위(={1/DR:.2f} 1/A)로 고정 \u2014 비닝과 무관, 원자거리(~1.6A) 분해 어려움.")
    print("  \u2192 거리/화합물 판별은 RDF(\u00a76b) 사용. cepstrum은 \u00a74 fluctuation \u00b7 \u00a75 상분리에 활용.")
n=mcep.shape[0]; ext=DR*(n//2)
fig,ax=plt.subplots(1,2,figsize=(12,4.4))
im=ax[0].imshow(mcep, cmap="inferno", extent=[-ext,ext,-ext,ext],
                vmax=np.percentile(mcep,99.5)); ax[0].set_title("mean EWPC (quefrency, Å)")
ax[0].set_xlabel("r_x (Å)"); ax[0].set_ylabel("r_y (Å)"); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[1].plot(r_ax, prof, "k-")
for (ri,ro),c in zip(BANDS, plt.cm.viridis(np.linspace(0,0.85,len(BANDS)))):
    ax[1].axvspan(ri,ro,color=c,alpha=0.12)
# 중심빔(디스크 경계) 인공물 예상 위치 = 1/q_beam (§2c). 이 근처 피크는 원자신호 아닐 수 있음.
try:
    if np.isfinite(r_beam) and r_beam <= r_ax.max():
        ax[1].axvline(r_beam, color="orange", ls="-.", lw=1.4)
        ax[1].text(r_beam, ax[1].get_ylim()[1]*0.6, f"beam-edge ~1/q_beam={r_beam:.1f}Å", rotation=90, fontsize=6, color="darkorange", va="center", ha="right")
except NameError:
    pass
for lbl,d in CEPSTRAL_REF.items():                 # pair 거리 → cepstral 간격 = d/1.23
    dc=d/EHRENFEST
    if dc<=min(6.5,r_ax.max()):
        ax[1].axvline(dc,color="0.5",ls=":",lw=0.8)
        ax[1].text(dc, ax[1].get_ylim()[1]*0.98, lbl.split()[0], rotation=90, fontsize=5, va="top", ha="right", color="0.4")
ax[1].set_xlim(0, 6.5)   # 구조 신호 구간만 (전체 quefrency는 매우 큼)
ax[1].set_xlabel("cepstral r (A) = spacing 1/q  (pair ~ x1.23; gray = pair/1.23)"); ax[1].set_ylabel("cepstral intensity")
ax[1].set_title("radial cepstral profile (peaks = interatomic distances)")
plt.tight_layout(); save(fig,"03_mean_ewpc_profile"); plt.show()
np.save(os.path.join(SAVE_DIR,"03_mean_ewpc.npy"), mcep)
save_csv("03_cepstral_radial_profile", ["r_A","cepstral_intensity"],
         [[f"{r_ax[i]:.4f}", f"{prof[i]:.6g}"] for i in range(len(r_ax))])



## 4) FC-STEM 이미지 — 거리 밴드별 fluctuation (혼합상 매핑)

각 거리 밴드에서 켑스트럼의 정규분산 $F(R_p)$ 를 스캔에 매핑. **밝음 = 큰 fluctuation**(그 거리 범위에서
질서/스펙클이 큼 → 결정질/특정 상). 밴드를 바꾸면 다른 상이 드러납니다.


In [ ]:

maps=fds.fluctuation_multiband(cube, BANDS, QPP, offset=EWPC_OFFSET, window=WINDOW,
                               n_jobs=N_JOBS, progress=True)
maps=[np.where(material, np.asarray(m), np.nan) for m in maps]   # 빈(진공) 위치 제외
nB=len(BANDS)
fig,ax=plt.subplots(1,nB,figsize=(3.2*nB,3.4),squeeze=False)
for j,((ri,ro),m) in enumerate(zip(BANDS,maps)):
    im=ax[0][j].imshow(as_map(m), cmap="viridis"); ax[0][j].set_title(f"F: {ri}-{ro} Å", fontsize=9)
    ax[0][j].axis("off"); plt.colorbar(im,ax=ax[0][j],fraction=0.046)
fig.suptitle("FC-STEM fluctuation images (bright = ordered in that distance band)", y=1.03)
plt.tight_layout(); save(fig,"04_fcstem_fluctuation"); plt.show()
save_csv("04_fcstem_fluctuation", ["scan_index"]+[f"F_{ri}_{ro}A" for ri,ro in BANDS],
         [[i]+[f"{np.asarray(m).ravel()[i]:.6g}" for m in maps] for i in range(np.asarray(maps[0]).size)])



## 5) 켑스트럼 프로파일 분해 (NMF k=4) — 상 분리

각 위치의 켑스트럼 방사 프로파일(원자간 거리 시그니처)을 분해. 켑스트럼은 **비음수**라 NMF가 유효합니다.
먼저 **성분 수를 데이터로 확인**(5a: elbow + 누적분산)하고, 그 수로 분해(5b)해서 **성분 피크를 알려진 Li
화합물 거리와 비교**합니다. → "K=3이 진짜인가"에 대한 객관적 근거.


In [ ]:

profs, r_p = fds.ewpc_profiles(matflat, QPP, r_min=0.4, offset=EWPC_OFFSET, window=WINDOW,
                               n_jobs=N_JOBS, progress=True)   # 물질 위치만
profs = np.clip(np.nan_to_num(profs), 0, None)                 # 비음수 보장

# --- 5a) 성분 수 확인: NMF 재구성 오차 + PCA 누적분산 vs K ---
Xn = np.linalg.norm(profs) + 1e-9
errs = []
for k in range(1, KMAX_SCAN+1):
    d = fds.decompose_profiles(profs, n_components=k, method="nmf", x=r_p)
    errs.append(float(getattr(d.model, "reconstruction_err_", np.nan)) / Xn)
pcp = fds.decompose_profiles(profs, n_components=KMAX_SCAN, method="pca", x=r_p)
cum = np.cumsum(pcp.explained_variance_ratio)
for fr in (0.95, 0.99):
    idx = np.where(cum >= fr)[0]
    if idx.size: print(f"cepstral profiles reach {int(fr*100)}% variance at K = {idx[0]+1}")
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(range(1,KMAX_SCAN+1), errs, "o-"); ax[0].set_xlabel("K"); ax[0].set_ylabel("relative recon. error")
ax[0].set_title("NMF error vs K (elbow ~ #structures)"); ax[0].axvline(K, color="r", ls=":", label=f"K={K}"); ax[0].legend()
ax[1].plot(range(1,KMAX_SCAN+1), cum, "s-"); ax[1].axhline(0.99, color="r", ls=":"); ax[1].axhline(0.95, color="orange", ls=":")
ax[1].set_ylim(0,1.02); ax[1].set_xlabel("K"); ax[1].set_ylabel("cumulative var"); ax[1].set_title("PCA cumulative variance")
plt.tight_layout(); save(fig,"05a_component_scan"); plt.show()

# --- 5b) K 성분 분해 + Li 화합물 거리 참고선 ---
METHOD = "nmf"
dp_c = fds.decompose_profiles(profs, n_components=K, method=METHOD, x=r_p)
frac = dp_c.fractions
fig,ax=plt.subplots(2,K,figsize=(3.4*K,6))
for i in range(K):
    ax[0,i].plot(r_p, dp_c.components[i], lw=1.4)
    for lbl,d in CEPSTRAL_REF.items():
        dc=d/EHRENFEST                                   # pair→cepstral 간격
        if dc <= min(6.5, r_p.max()):
            ax[0,i].axvline(dc, color="0.6", ls=":", lw=0.8)
            if i==0: ax[0,i].text(dc, ax[0,i].get_ylim()[1]*0.98, lbl.split()[0], rotation=90, fontsize=5, va="top", ha="right", color="0.4")
    ax[0,i].set_xlim(0, 6.5)
    ax[0,i].set_title(f"comp {i+1}", fontsize=9); ax[0,i].set_xlabel("cepstral r (A) = spacing (pair/1.23)")
    im=ax[1,i].imshow(scatter(frac[:,i]), cmap="viridis"); ax[1,i].set_title(f"fraction {i+1}", fontsize=9); ax[1,i].axis("off")
fig.suptitle(f"cepstral {METHOD.upper()} (k={K}) — peaks vs Li-compound distances", y=1.02)
plt.tight_layout(); save(fig,"05_cepstral_profile_nmf"); plt.show()
save_csv("05_cepstral_components", ["r_A"]+[f"comp{i+1}" for i in range(K)],
         [[f"{r_p[j]:.4f}"]+[f"{dp_c.components[i][j]:.6g}" for i in range(K)] for j in range(len(r_p))])
print("참고 거리(Å):", CEPSTRAL_REF)



## 6) 켑스트럼 상(相)별 RDF — 주요 peak / 물질 확인

§5 NMF 분율의 **argmax로 각 위치를 한 상에 배정**(K개 영역) → 각 영역의 패턴을 **빔 중심 정렬 후 평균**
(`average_pattern_aligned`, wander 제거) → **RDF**. 상별로 어떤 원자간 거리(peak)가 나오는지 보고, Li
화합물 참고선과 비교해 **각 상이 어떤 물질인지** 추정합니다.


In [ ]:
labels = dp_c.fractions.argmax(1)                    # 물질 위치별 상 라벨(0..K-1)
# --- 진단: K개 성분 중 실제로 위치를 '차지한' 상은 몇 개인가 (hard argmax) ---
counts = np.bincount(labels, minlength=K)
fmean  = dp_c.fractions.mean(0)
print('phase population (argmax 승리 위치 수):', {k:int(counts[k]) for k in range(K)})
print('mean fraction/comp            :', {k:round(float(fmean[k]),3) for k in range(K)})
alive = int((counts >= 5).sum())
if alive < K:
    dead=[k for k in range(K) if counts[k] < 5]
    print(f"\u26a0 K={K}로 분해했지만 지도에 실제로 나타난 상은 {alive}개뿐 — 상 {dead}는 argmax를 거의 못 이김.")
    print(f"  \u2192 데이터가 지지하는 구조 수 \u2248 {alive}. K를 {alive}로 줄이거나 §5a elbow와 대조하세요.")

labfull = np.full(KEEP.size, -1, int); labfull[KEEP] = labels
labmap = labfull.reshape(scan)
tgt = (dp[1]/2.0, dp[0]/2.0)                          # 정렬 목표 = 검출기 중심
beam = max(2, int(CFG_RDF.q_int_min/QPP))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
im=ax[0].imshow(np.where(labmap>=0, labmap, np.nan), cmap="tab10", vmin=0, vmax=K-1); ax[0].set_title(f"phase label map (K={K}, alive={alive})")
ax[0].axis("off"); plt.colorbar(im, ax=ax[0], fraction=0.046, ticks=range(K))
rdf_rows=[]
for k in range(K):
    region = (labmap == k)
    if region.sum() < 5: continue
    pat = fds.average_pattern_aligned(cube, region, target=tgt, threshold=0.3)
    pat = fds.clean_pattern(pat, hot_threshold=HOT_THRESHOLD)     # 고정 hot/dead 정리
    rr = fds.pattern_to_rdf(pat, QPP, CFG_RDF, center=tgt, center_beam_radius=beam)
    ax[1].plot(rr.r, rr.Gr, lw=1.4, label=f"phase {k} ({100*region.mean():.0f}%)")
    rdf_rows.append((k, rr))
for lbl,d in CEPSTRAL_REF.items():
    if d<=8: ax[1].axvline(d,color="0.6",ls=":",lw=0.8); ax[1].text(d, ax[1].get_ylim()[1]*0.98, lbl.split()[0], rotation=90, fontsize=5, va="top", ha="right", color="0.4")
ax[1].axhline(0,color="0.85",lw=.8); ax[1].set_xlim(0,8); ax[1].set_xlabel("r (Å)"); ax[1].set_ylabel("G(r)")
ax[1].set_title("per-phase RDF (aligned avg) vs Li-compound distances"); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,"06_phase_rdf"); plt.show()
if rdf_rows:
    r0=rdf_rows[0][1].r
    save_csv("06_phase_rdf", ["r_A"]+[f"phase{k}" for k,_ in rdf_rows],
             [[f"{r0[j]:.4f}"]+[f"{rr.Gr[j]:.6g}" for _,rr in rdf_rows] for j in range(len(r0))])
print("\nAll outputs in:", os.path.abspath(SAVE_DIR))


## 6b) 후보 화합물 지문(1st + **2nd peak**) 비교 — 어떤 Li 상인가

1st peak만으론 **LiF·Li2O·Li3N·Li2CO3가 모두 ~2.0 Å**로 겹칩니다. 구별의 핵심은 **2nd peak**과
그 **비율 r₂/r₁**(구조 지문)입니다:

| 화합물 | 구조 | 1st (양이온–음이온) | 2nd (음이온–음이온) | **r₂/r₁** |
|---|---|---|---|---|
| **LiF** | rocksalt | 2.01 (Li–F) | 2.85 (F–F) | **1.41** (√2) |
| **Li2O** | antifluorite | 2.00 (Li–O) | 3.27 (O–O) | **1.63** |
| **Li3N** | hex | 1.94 (Li–N) | 3.65 (N–N) | **1.88** |
| **Li2S** | antifluorite | 2.48 (Li–S) | 4.04 (S–S) | **1.63** |
| **Li2CO3** | 분자성 | **1.28 (C–O)** · 2.0 (Li–O) | 2.22 (O–O) | — |

판별 포인트: **① Li2S**는 1st가 2.48로 홀로 큼. **② Li2CO3**는 ~1.28 Å에 **짧은 C–O**(다른 상엔 없음).
**③ LiF vs Li2O/Li3N**는 2nd(또는 r₂/r₁)로 갈림. **r₂/r₁은 q 스케일(캘리브레이션)에 무관**하므로
캘리브레이션이 불확실해도 이 비율은 신뢰할 수 있습니다.

In [ ]:
# 후보 Li 상의 지문 거리(Å, 결정 근사). 필요시 수정.
COMPOUND_REF = {
  'LiF'   : dict(r1=2.01, r2=2.85, ratio=1.41, color='#e41a1c', marks={'Li-F':2.01,'F-F':2.85,'Li-F2':3.49}),
  'Li2O'  : dict(r1=2.00, r2=3.27, ratio=1.63, color='#377eb8', marks={'Li-O':2.00,'Li-Li':2.31,'O-O':3.27}),
  'Li3N'  : dict(r1=1.94, r2=3.65, ratio=1.88, color='#4daf4a', marks={'Li-N':1.94,'Li-N2':2.11,'N-N':3.65}),
  'Li2CO3': dict(r1=1.28, r2=2.22, ratio=1.73, color='#984ea3', marks={'C-O':1.28,'Li-O':2.00,'O-O':2.22}),
  'Li2S'  : dict(r1=2.48, r2=4.04, ratio=1.63, color='#ff7f00', marks={'Li-S':2.48,'Li-Li':2.86,'S-S':4.04}),
}
def _phase_peaks(r, G):
    '''물질 상 G(r)의 1st/2nd peak(r>1.15 Å, prominence 기준).'''
    m = r > 1.15
    pk = fds.find_peaks_1d(r[m], G[m], prominence=0.05*np.nanmax(np.abs(G[m])), distance=max(3,int(0.35/(r[1]-r[0]))))
    pk = sorted(pk, key=lambda d: d['x'])            # r 오름차순
    rs = [d['x'] for d in pk]
    r1 = rs[0] if rs else np.nan
    r2 = rs[1] if len(rs) > 1 else np.nan
    short = any(d['x'] < 1.55 for d in pk)           # ~1.28 C-O 존재?
    return r1, r2, short, pk

print('phase | r1(A)  r2(A)  r2/r1 | 짧은C-O? | best(비율) | best(절대)')
print('-'*74)
phase_fp=[]
for k, rr in rdf_rows:
    r1,r2,short,pk = _phase_peaks(rr.r, rr.Gr)
    ratio = r2/r1 if (np.isfinite(r1) and np.isfinite(r2) and r1>0) else np.nan
    # 비율 매칭(캘리브레이션 무관): r2/r1이 가장 가까운 화합물
    by_ratio = min(COMPOUND_REF, key=lambda c: abs(COMPOUND_REF[c]['ratio']-ratio)) if np.isfinite(ratio) else '-'
    # 절대 매칭: (r1,r2) 유클리드 거리
    def _d(c):
        v=COMPOUND_REF[c]; s=0; n=0
        if np.isfinite(r1): s+=(v['r1']-r1)**2; n+=1
        if np.isfinite(r2): s+=(v['r2']-r2)**2; n+=1
        return s/max(n,1)
    by_abs = min(COMPOUND_REF, key=_d) if np.isfinite(r1) else '-'
    if short: by_abs='Li2CO3?'                        # C-O 지문 우선
    phase_fp.append((k,r1,r2,ratio,short,by_ratio,by_abs))
    print(f'  {k}   | {r1:5.2f}  {r2:5.2f}  {ratio:5.2f} |   {"Y" if short else "-"}    | {by_ratio:9s} | {by_abs}')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
# (좌) 상별 RDF + 화합물 지문선(색=화합물, 실선=1st, 파선=2nd/기타)
for k, rr in rdf_rows:
    ax[0].plot(rr.r, rr.Gr, lw=1.5, label=f'phase {k}')
for c,v in COMPOUND_REF.items():
    for name,d in v['marks'].items():
        if d<=8: ax[0].axvline(d, color=v['color'], ls='-' if abs(d-v['r1'])<1e-6 else ':', lw=1.0, alpha=0.55)
    ax[0].plot([],[],color=v['color'],lw=2,label=f"{c} (r2/r1={v['ratio']})")
ax[0].axhline(0,color='0.85',lw=.8); ax[0].set_xlim(0,7); ax[0].set_xlabel('r (Å)'); ax[0].set_ylabel('G(r)')
ax[0].set_title('per-phase RDF vs compound fingerprints'); ax[0].legend(fontsize=7, ncol=2)
# (우) r2/r1 비율 비교 — 캘리브레이션 무관 지문
for c,v in COMPOUND_REF.items():
    ax[1].axhline(v['ratio'], color=v['color'], ls='--', lw=1, alpha=0.7)
    ax[1].text(len(phase_fp)-0.4, v['ratio'], c, color=v['color'], fontsize=8, va='center')
for i,(k,r1,r2,ratio,short,br,ba) in enumerate(phase_fp):
    if np.isfinite(ratio): ax[1].scatter(i, ratio, s=90, zorder=5, edgecolor='k'); ax[1].text(i, ratio+0.03, f'p{k}', ha='center', fontsize=8)
ax[1].set_xlim(-0.6,len(phase_fp)+0.3); ax[1].set_xticks(range(len(phase_fp))); ax[1].set_xticklabels([f'phase {k}' for k,*_ in phase_fp])
ax[1].set_ylabel('r2 / r1  (structure fingerprint, calibration-free)'); ax[1].set_title('measured ratio vs compounds')
plt.tight_layout(); save(fig,'06b_compound_fingerprint'); plt.show()
save_csv('06b_fingerprint', ['phase','r1_A','r2_A','r2_over_r1','short_CO','best_by_ratio','best_by_abs'],
         [[k,f'{r1:.3f}',f'{r2:.3f}',f'{ratio:.3f}',int(short),br,ba] for k,r1,r2,ratio,short,br,ba in phase_fp])


## 6c) **cepstrum** 도메인 지문 + RDF와 나란히 비교

§6b는 **RDF(G(r))** 로 비교했습니다. 여기선 **같은 상 라벨**로 **cepstrum**(§5 프로파일)에서도 1st/2nd
peak을 뽑아 비교합니다. 약한 신호에서는 배경제거가 필요없는 cepstrum이 더 안정적일 수 있습니다.

**중요(스케일 차이)**: 같은 링이라도 cepstrum peak ≈ `1/q`(링 간격), RDF peak ≈ `1.23/q`(pair 거리).
→ 절대거리는 **×1.23(Ehrenfest)** 로 이어주고, **r₂/r₁ 비율은 이 인자가 상쇄**되어 cepstrum·RDF·참고표가
모두 동일 스케일이 됩니다. 그래서 **비율 매칭이 캘리브레이션·방법에 무관한 가장 강건한 지표**입니다.
**두 방법(RDF·cepstrum)이 같은 화합물을 가리키면 신뢰도가 크게 올라갑니다.**

In [ ]:
# cepstrum peak(링 간격 1/q) → pair 거리(1.23/q) 브릿지. 비율은 상쇄되어 동일.
EHRENFEST = globals().get('EHRENFEST', 1.23)
# labels(§6)와 profs(§5) 행은 1:1 정렬(둘 다 물질 위치 순서).
print('phase | cep r1,r2(Å)  r2/r1 | ~pair ×1.23 | cep best(비율) || RDF best(비율) | 일치?')
print('-'*82)
comb=[]
for k, rr in rdf_rows:
    sel = (labels == k)
    if sel.sum() < 3: continue
    cep = profs[sel].mean(0)
    r1c, r2c, shortc, _ = _phase_peaks(r_p, cep)
    ratioc = r2c/r1c if (np.isfinite(r1c) and np.isfinite(r2c) and r1c>0) else np.nan
    by_ratio_c = min(COMPOUND_REF, key=lambda c: abs(COMPOUND_REF[c]['ratio']-ratioc)) if np.isfinite(ratioc) else '-'
    fp = next((x for x in phase_fp if x[0]==k), None)          # §6b의 RDF 결과
    rdf_ratio = fp[3] if fp else np.nan; rdf_best = fp[5] if fp else '-'
    agree = 'AGREE' if (by_ratio_c==rdf_best and by_ratio_c!='-') else ''
    comb.append((k, r1c, r2c, ratioc, by_ratio_c, rdf_ratio, rdf_best, agree))
    print(f'  {k}   | {r1c:4.2f},{r2c:4.2f}   {ratioc:5.2f} | {EHRENFEST*r1c:4.2f},{EHRENFEST*r2c:4.2f} | {by_ratio_c:9s} || {rdf_best:9s} ({rdf_ratio:4.2f}) | {agree}')

if comb and all(not np.isfinite(x[3]) for x in comb):
    print("\n\u26a0 cepstral peak 미분해(프로파일 단조) \u2192 cepstrum 지문 신뢰 낮음. RDF(\u00a76b) r2/r1 비율을 기준으로 판단하세요.")

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
# (좌) 상별 cepstrum 프로파일 + 화합물 지문선(cepstral 스케일 = pair/1.23)
for k, rr in rdf_rows:
    sel = (labels == k)
    if sel.sum() >= 3: ax[0].plot(r_p, profs[sel].mean(0), lw=1.5, label=f'phase {k}')
for c,v in COMPOUND_REF.items():
    ax[0].axvline(v['r1']/EHRENFEST, color=v['color'], ls='-', lw=1.0, alpha=0.55)
    ax[0].axvline(v['r2']/EHRENFEST, color=v['color'], ls=':', lw=1.0, alpha=0.55)
    ax[0].plot([],[],color=v['color'],lw=2,label=f"{c} (r2/r1={v['ratio']})")
ax[0].set_xlim(0, 6.5); ax[0].set_xlabel('cepstral r (A) = spacing (x1.23 ~ pair dist)'); ax[0].set_ylabel('cepstral intensity')
ax[0].set_title('per-phase CEPSTRUM vs compound fingerprints'); ax[0].legend(fontsize=7, ncol=2)
# (우) r2/r1 비율: cepstrum(●) vs RDF(◆) — 같은 상이면 같은 색, 화합물 밴드와 비교
for c,v in COMPOUND_REF.items():
    ax[1].axhline(v['ratio'], color=v['color'], ls='--', lw=1, alpha=0.7)
    ax[1].text(len(comb)-0.4, v['ratio'], c, color=v['color'], fontsize=8, va='center')
for i,(k,r1c,r2c,ratioc,brc,rr_ratio,rr_best,ag) in enumerate(comb):
    if np.isfinite(ratioc):  ax[1].scatter(i-0.09, ratioc, s=90, marker='o', zorder=5, edgecolor='k', label='cepstrum' if i==0 else None)
    if np.isfinite(rr_ratio):ax[1].scatter(i+0.09, rr_ratio, s=90, marker='D', zorder=5, edgecolor='k', label='RDF' if i==0 else None)
    ax[1].text(i, max(filter(np.isfinite,[ratioc,rr_ratio]),default=1.5)+0.05, f'p{k}', ha='center', fontsize=8)
ax[1].set_xlim(-0.6,len(comb)+0.3); ax[1].set_xticks(range(len(comb))); ax[1].set_xticklabels([f'phase {k}' for k,*_ in comb])
ax[1].set_ylabel('r2 / r1  (calibration & method free)'); ax[1].set_title('ratio: cepstrum vs RDF vs compounds'); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,'06c_cepstrum_fingerprint'); plt.show()
save_csv('06c_fingerprint_both', ['phase','cep_r1','cep_r2','cep_ratio','cep_best','rdf_ratio','rdf_best','agree'],
         [[k,f'{r1c:.3f}',f'{r2c:.3f}',f'{ratioc:.3f}',brc,f'{rr_ratio:.3f}',rr_best,ag] for k,r1c,r2c,ratioc,brc,rr_ratio,rr_best,ag in comb])


## 6d) 후보 Li 화합물로 supervised 언믹싱 (NNLS) — 상별 함량

임의 NMF 대신 **아는 5개 화합물 signature**(결정 거리로 합성한 RDF)를 기저로 각 상의 RDF를 맞춥니다.
없는 화합물은 함량 ≈ 0. 먼저 **참조 분리도**(1=구별 불가)를 출력 — 이 분해능(Δr~1 Å)에선
**LiF·Li2O·Li3N의 1st peak이 겹쳐** 서로 구별이 약하고(합쳐서 해석), **Li2S(2.48)·Li2CO3(1.28)** 는
잘 구별됩니다. `fit_res`가 작을수록 그 상이 후보 화합물들로 잘 설명됨.

In [ ]:
CANDIDATES = ['LiF','Li2O','Li3N','Li2CO3','Li2S']       # 후보(빼거나 추가 가능)
qmax_eff = min(CFG_RDF.q_int_max, QPP*(dp[0]//2))
REF_SIGMA = float(np.clip(0.5/max(qmax_eff,0.4), 0.25, 0.8))   # 데이터 분해능(≈0.5/q_max)에 맞춘 브로드닝
if rdf_rows:
    r0 = rdf_rows[0][1].r
    refs = fds.build_references(r0, compounds=CANDIDATES, sigma=REF_SIGMA)
    nms, C = fds.reference_degeneracy(refs)
    print(f"REF_SIGMA={REF_SIGMA:.2f} A | 참조 분리도(1=구별불가):")
    print('         '+' '.join(f'{n:>7s}' for n in nms))
    for i,n in enumerate(nms): print(f'{n:>7s} '+' '.join(f'{C[i,j]:7.2f}' for j in range(len(nms))))
    hard=[(nms[i],nms[j]) for i in range(len(nms)) for j in range(i+1,len(nms)) if C[i,j]>0.9]
    if hard: print('  \u26a0 이 분해능에서 사실상 구별 불가 쌍:', hard, '\u2192 함량 분배 신뢰 낮음(합쳐 해석).')

    stack = np.vstack([rr.Gr for _,rr in rdf_rows])
    names, AB, resid = fds.unmix_nnls(stack, refs, r=r0, r_range=(CFG_RDF.r_min, 6.0))
    AB = np.atleast_2d(AB); resid = np.atleast_1d(resid)
    _r1s=[x[1] for x in phase_fp if np.isfinite(x[1])]
    if _r1s and np.median(_r1s) < 1.85:
        print(f"\u26a0 측정 1st peak 중앙값 {np.median(_r1s):.2f} A < 예상 ~2.0 \u2192 CALIBRATION 미적용 가능.")
        print("   \u2192 §1에서 CALIB_R_TARGET=2.0 두고 §2c부터 재실행하세요. 지금 언믹싱은 저-r 화합물(Li2CO3)로 치우칩니다.")
        print("   (캘리브레이션 무관한 §6b r2/r1 비율을 우선 신뢰하세요.)")
    print('\nphase | ' + ' '.join(f'{n:>7s}' for n in names) + ' | fit_res')
    for (k,_),ab,rs in zip(rdf_rows, AB, resid):
        print(f'  {k}   | ' + ' '.join(f'{a:7.2f}' for a in ab) + f' | {rs:.3f}')

    fig,ax=plt.subplots(1,2,figsize=(13,4.6))
    for c in names: ax[0].plot(r0, refs[c], lw=1.3, label=c)
    for k,rr in rdf_rows:
        g=np.clip(rr.Gr,0,None); g=g/(np.linalg.norm(g)+1e-9); ax[0].plot(rr.r, g, 'k-', lw=0.6, alpha=0.35)
    ax[0].set_xlim(0,6.5); ax[0].set_xlabel('r (Å)'); ax[0].set_ylabel('normalized')
    ax[0].set_title(f'compound refs (σ={REF_SIGMA:.2f}) vs phase RDFs (gray)'); ax[0].legend(fontsize=8)
    x=np.arange(len(rdf_rows)); w=0.8/len(names)
    for j,c in enumerate(names): ax[1].bar(x+j*w, AB[:,j], w, label=c)
    ax[1].set_xticks(x+0.4-w/2); ax[1].set_xticklabels([f'p{k}' for k,_ in rdf_rows])
    ax[1].set_ylabel('abundance (fraction)'); ax[1].set_title('per-phase compound abundance (NNLS)'); ax[1].legend(fontsize=8)
    plt.tight_layout(); save(fig,'06d_compound_unmix'); plt.show()
    save_csv('06d_compound_unmix', ['phase']+names+['fit_res'],
             [[k]+[f'{AB[i,j]:.4f}' for j in range(len(names))]+[f'{resid[i]:.4f}'] for i,(k,_) in enumerate(rdf_rows)])



**정리** — EWPC(로그→역FFT)로 배경 제거 없이 원자간 거리 신호를 얻고, (3) 평균 프로파일, (4) 거리 밴드별
FC-STEM fluctuation 매핑, (5) 프로파일 NMF k=4 로 혼합 비정질상을 분리합니다.
- **거리 밴드(`BANDS`)**: §3 프로파일에서 상별로 두드러지는 거리 구간을 골라 넣으세요.
- 위치가 많으면 (3)(4)(5)가 병렬로 수 분 걸릴 수 있습니다(`N_JOBS`, `DET_BIN`).
- 켑스트럼 거리축은 `dr=1/(N·q_per_px)`. 프로파일 피크가 예상 Å와 다르면 `Q_UNIT_HINT`/캘리브레이션 확인.
- 결정+비정질 혼합에 특히 강합니다(결정=밴드에서 밝음). 약한 리튬화합물 신호에 RDF 대안으로 권장.
